# baseline v3

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

Colab의 GPU 환경에서 개발되었습니다.
- 런타임 - 런타임 유형 변경 - GPU로 변경(T4 GPU 등)



# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 최신 버전으로 라이브러리를 업데이트합니다.

- 아래 셀 실행
- 실행 완료 후 런타임 - 세션 다시 시작

In [ ]:
!pip -q uninstall -y transformers
!pip -q install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip -q install --force-reinstall --no-deps "transformers==4.57.6"
!pip -q install "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.3" datasets "pillow<11" pandas --upgrade

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # 메모리 파편화로 인한 OOM 방지

import torch
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())

# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

본 베이스라인은 colab에서 구글 드라이브를 마운트하여 사용합니다.

데이터를 압축 해제하는데 몇 분 정도의 시간이 소요됩니다.

#### 실습 참고 내용

    챕터 2-2 합성 데이터 실습
    - 구글 드라이브 마운트 : drive()

In [ ]:
# 구글드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 압축 해제
!unzip "/content/drive/My Drive/2026-ssafy-15-2-ai.zip" -d "/content/"

# 라이브러리, 데이터, 설정

In [ ]:
import os, re, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm

# 이미지 로드 시 픽셀 제한 해제
Image.MAX_IMAGE_PIXELS = None

# 디바이스 GPU 우선 사용 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# 사전 학습 모델 정의
# A100 40GB 환경: 양자화 없이 bf16 그대로 로드 (정밀도 손실 방지, 데스크탑 16GB 버전과 의도적으로 다름)
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
IMAGE_SIZE = 384
MAX_NEW_TOKENS = 8
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# 실험 설정
EPOCHS = 3             # A100에서 학습이 훨씬 빨라져 epoch를 늘림 - 드라이런으로 재조정 가능
LORA_R = 32            # 40GB 여유를 활용해 rank 상향 (표현력 확대 -> 정확도 개선)
BATCH_SIZE = 2         # OOM 재발 방지를 위해 4->2로 하향 (gradient checkpointing과 함께 안전 마진 확보)
MIN_DEV_VOTES = 3      # dev pseudo-label 다수결 임계값 (5표 중 3표 이상)
DEV_AUGMENT = True
DEBUG_SUBSET = False   # True로 켜면 소규모 드라이런
DEBUG_N = 300

# 데이터셋 로드
train_df = pd.read_csv("/content/train.csv")
test_df  = pd.read_csv("/content/test.csv")
dev_df   = pd.read_csv("/content/dev.csv")

# 드라이런용 소규모 샘플링 (기본값은 전체 데이터 사용)
if DEBUG_SUBSET:
    train_df = train_df.sample(n=DEBUG_N, random_state=SEED).reset_index(drop=True)

In [ ]:
# dev.csv의 answer1~5 다수결로 pseudo-label 생성 (규정상 dev 데이터 증강 허용)
from collections import Counter

def majority_vote_answer(row, ans_cols=("answer1","answer2","answer3","answer4","answer5")):
    # 결측치(NaN)나 a/b/c/d가 아닌 값은 투표에서 제외
    valid = [str(row[c]).strip().lower() for c in ans_cols if pd.notna(row[c])]
    valid = [v for v in valid if v in ("a","b","c","d")]
    if not valid:
        return None, 0  # 유효 응답이 하나도 없으면 vote_count=0 -> min_votes 필터에서 자동 제외
    counts = Counter(valid)
    top = max(counts.values())
    winners = sorted(k for k, v in counts.items() if v == top)  # 동률시 사전순으로 결정
    return winners[0], top

def build_dev_pseudo_df(dev_df, min_votes=MIN_DEV_VOTES):
    ans_cols = ["answer1","answer2","answer3","answer4","answer5"]
    votes = dev_df.apply(lambda r: majority_vote_answer(r, ans_cols), axis=1, result_type="expand")
    votes.columns = ["answer", "vote_count"]
    out = pd.concat([dev_df.drop(columns=ans_cols), votes], axis=1)
    out = out[out["vote_count"] >= min_votes].reset_index(drop=True)
    return out[["id","path","question","a","b","c","d","answer"]]  # train_df와 동일 스키마

dev_pseudo_df = build_dev_pseudo_df(dev_df, MIN_DEV_VOTES) if DEV_AUGMENT else dev_df.iloc[0:0]
print(f"dev pseudo-label 채택: {len(dev_pseudo_df)} / {len(dev_df)}")

# 모델, Processor

이미 4bit로 양자화된 체크포인트(약 7.2GB)를 다운로드합니다. 10~20분 정도가 소요됩니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [ ]:
# 프로세서
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE*IMAGE_SIZE,
    max_pixels=IMAGE_SIZE*IMAGE_SIZE,
    trust_remote_code=True,
)

# 사전학습 모델 (A100 40GB: 양자화 없이 bf16 그대로 로드 - 정밀도 손실 방지)
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

# gradient checkpointing 활성화 (OOM 방지용 안전 마진 확보 - 속도 20~30% 손해를 감수하고 메모리 절약)
base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
base_model.enable_input_require_grads()

# LoRA 세팅
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_R*2,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)

# PEFT 모델 생성
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [ ]:
# 모델 지시사항
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

# 프롬프트
def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [ ]:
# 선지 순서 셔플 (위치 편향 제거용 증강)
def shuffle_mc(a, b, c, d, gold_letter, rng):
    letters = ["a","b","c","d"]
    values = [a, b, c, d]
    gold_idx = letters.index(gold_letter)
    idx_order = [0,1,2,3]
    rng.shuffle(idx_order)
    shuffled = [values[i] for i in idx_order]
    return (*shuffled, letters[idx_order.index(gold_idx)])

# 커스텀 데이터셋
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True, shuffle_choices=False, seed=SEED):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train
        self.shuffle_choices = shuffle_choices
        self.rng = random.Random(seed)

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        gold = str(row["answer"]).strip().lower() if self.train else None

        if self.train and self.shuffle_choices:
            a, b, c, d, gold = shuffle_mc(a, b, c, d, gold, self.rng)

        user_text = build_mc_prompt(q, a, b, c, d)

        # prompt_messages: 정답이 빠진 상태 (system+user까지) - loss 마스킹 경계 계산용
        prompt_messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}
            ]}
        ]
        messages = list(prompt_messages)
        if self.train:
            messages.append({"role":"assistant","content":[{"type":"text","text":gold}]})

        return {"messages": messages, "prompt_messages": prompt_messages, "image": img}

# 데이터 콜레이터
@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images, prompt_texts = [], [], []
        for sample in batch:
            messages = sample["messages"]
            img = sample["image"]

            text = self.processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
            texts.append(text)
            images.append(img)

            if self.train:
                prompt_text = self.processor.apply_chat_template(
                    sample["prompt_messages"],
                    tokenize=False,
                    add_generation_prompt=True
                )
                prompt_texts.append(prompt_text)

        enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt"
        )

        if self.train:
            enc["labels"] = enc["input_ids"].clone()
            # 배치 내 패딩 토큰은 loss 계산에서 제외
            enc["labels"][enc["attention_mask"] == 0] = -100
            # system+user(질문/이미지) 구간도 loss에서 제외 - 정답 글자(+종료 토큰)만 학습 신호로 사용
            for i, (prompt_text, img) in enumerate(zip(prompt_texts, images)):
                prompt_len = self.processor(text=[prompt_text], images=[img], return_tensors="pt")["input_ids"].shape[1]
                enc["labels"][i, :prompt_len] = -100

        return enc


# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [ ]:
# 검증용 데이터 분리 (gold label 기준으로 먼저 분리 - dev pseudo-label 병합 전)
split = int(len(train_df)*0.9)
train_subset, valid_subset = train_df.iloc[:split].copy(), train_df.iloc[split:].copy()

# dev pseudo-label 증강: train 쪽에만 병합 (valid는 순수 gold label로 유지해 신뢰도 확보)
if DEV_AUGMENT and len(dev_pseudo_df) > 0:
    train_subset = pd.concat([train_subset, dev_pseudo_df], ignore_index=True)
print(f"train_subset: {len(train_subset)} (dev 증강 포함), valid_subset: {len(valid_subset)}")

# VQAMCDataset 형태로 변환 (train에만 선지 셔플 적용)
train_ds = VQAMCDataset(train_subset, processor, train=True, shuffle_choices=True, seed=SEED)
valid_ds = VQAMCDataset(valid_subset, processor, train=True, shuffle_choices=False, seed=SEED)

# 데이터로더 (A100 40GB: 실배치 크기 BATCH_SIZE 사용)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=DataCollator(processor, True), num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=DataCollator(processor, True), num_workers=0)

# fine-tuning

- 전체 데이터 + dev 증강 데이터로 학습 : DEBUG_SUBSET으로 먼저 드라이런 권장

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
from tqdm.auto import tqdm

model = model.to(device)
GRAD_ACCUM = 1  # 이미 실배치(BATCH_SIZE)를 쓰므로 추가 누적은 최소화
SAVE_DIR = "/content/qwen3_vl_8b_lora"
SAVE_EVERY_N_STEPS = 500  # 에폭 중간 크래시(OOM 등) 대비 - 진행상황 주기적 저장

# 옵티마이저, 학습 스케줄러
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
num_training_steps = EPOCHS * math.ceil(len(train_loader)/GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(optimizer, int(num_training_steps*0.03), num_training_steps)

# 스케일러
scaler = torch.amp.GradScaler('cuda', enabled=True)

# 학습 루프
global_step = 0
for epoch in range(EPOCHS):
    running = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")
    for step, batch in enumerate(progress_bar, start=1):
        batch = {k:v.to(device) for k,v in batch.items()}
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()
        del batch, outputs, loss

        if step % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            avg_loss = running / GRAD_ACCUM
            progress_bar.set_postfix({"loss": f"{avg_loss:.3f}"})
            running = 0.0

            # 에폭 중간 진행상황 저장 (OOM/크래시로 통째로 날아가는 것 방지)
            if global_step % SAVE_EVERY_N_STEPS == 0:
                step_dir = f"{SAVE_DIR}_epoch{epoch+1}_step{global_step}"
                model.save_pretrained(step_dir)
                print("Saved mid-epoch checkpoint:", step_dir)
                torch.cuda.empty_cache()

    model.eval()
    val_loss = 0.0
    val_steps = 0
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid]", unit="batch"):
            vb = {k:v.to(device) for k,v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1
    print(f"[Epoch {epoch+1}] valid loss {val_loss/val_steps:.4f}")
    model.train()

    # epoch마다 체크포인트 저장 (장시간 학습 중 크래시 대비)
    epoch_dir = f"{SAVE_DIR}_epoch{epoch+1}"
    model.save_pretrained(epoch_dir)
    processor.save_pretrained(epoch_dir)
    print("Saved checkpoint:", epoch_dir)

# 최종 모델 저장
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print("Saved:", SAVE_DIR)


# inference

모델 크기(8B)와 전체 데이터 학습 규모를 고려하면 소요 시간이 기존 베이스라인 대비 크게 늘어날 수 있습니다.
`DEBUG_SUBSET=True`로 먼저 소규모 드라이런을 돌려 실제 소요 시간을 측정한 뒤 전체 실행 여부를 판단하세요.

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
# 학습 중 사용된 optimizer/scheduler/scaler를 정리해 추론 전에 VRAM을 확보
import gc
for _name in ["optimizer", "scheduler", "scaler"]:
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()
print("추론 전 VRAM 확보 완료. allocated GB:", torch.cuda.memory_allocated()/1e9)

# a/b/c/d 각 글자에 대응하는 토큰 id를 학습 때와 동일한 방식으로 산출
def build_letter_token_ids(processor):
    dummy_messages = [
        {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
        {"role":"user","content":[{"type":"text","text":"dummy"}]},
    ]
    prefix_text = processor.apply_chat_template(dummy_messages, tokenize=False, add_generation_prompt=True)
    prefix_ids = processor.tokenizer(prefix_text, add_special_tokens=False)["input_ids"]
    letter_ids = {}
    for letter in ["a","b","c","d"]:
        full_ids = processor.tokenizer(prefix_text + letter, add_special_tokens=False)["input_ids"]
        assert full_ids[:len(prefix_ids)] == prefix_ids, f"template boundary mismatch for {letter!r}"
        letter_ids[letter] = full_ids[len(prefix_ids):][0]
    return letter_ids

model.eval()
letter_ids = build_letter_token_ids(processor)
letters_order = ["a","b","c","d"]
letter_id_tensor = torch.tensor([letter_ids[l] for l in letters_order], device=device)
print("letter -> token id:", letter_ids)
for l, tid in letter_ids.items():
    print(f"  sanity check decode({tid}) = {processor.tokenizer.decode([tid])!r}")

# 배치 추론: generate() 대신 마지막 위치의 로짓에서 a/b/c/d 4개 토큰의 확률만 비교
INFER_BATCH_SIZE = 8  # OOM 시 4, 2, 1 순으로 낮추세요
processor.tokenizer.padding_side = "left"  # 배치 내 모든 샘플의 마지막 위치가 '다음 토큰 생성 직전'이 되도록

preds = []
with torch.no_grad():
    for start in tqdm(range(0, len(test_df), INFER_BATCH_SIZE), desc="Inference", unit="batch"):
        chunk = test_df.iloc[start:start+INFER_BATCH_SIZE]
        texts, images = [], []
        for _, row in chunk.iterrows():
            img = Image.open(row["path"]).convert("RGB")
            user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])

            messages = [
                {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
                {"role":"user","content":[
                    {"type":"image","image":img},
                    {"type":"text","text":user_text}
                ]}
            ]
            texts.append(processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
            images.append(img)

        inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(device)

        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            out = model(**inputs)
        last_logits = out.logits[:, -1, :]
        best_idx = last_logits[:, letter_id_tensor].argmax(dim=-1).tolist()
        preds.extend(letters_order[i] for i in best_idx)
        del inputs, out, last_logits

processor.tokenizer.padding_side = "right"

# 제출 파일 생성
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("/content/submission.csv", index=False)
print("Saved /content/submission.csv")

In [ ]:
# 예측 분포 확인 (특정 글자로 쏠리지 않는지 sanity check)
print(submission.head())
print(submission["answer"].value_counts())

# 이어서 학습하기 (resume)

가장 성능이 좋았던 체크포인트(epoch3)에서 추가로 epoch를 더 학습합니다.

- optimizer/scheduler 상태는 저장해두지 않았으므로 새로 생성합니다 (학습률은 이미 수렴된 지점에서 이어가는 것이라 처음보다 낮게 설정)
- `RESUME_FROM`을 원하는 체크포인트 경로로, `ADDITIONAL_EPOCHS`를 원하는 추가 epoch 수로 바꿔서 사용하세요

In [ ]:
import gc
from peft import PeftModel

# 기존 모델/옵티마이저 정리 (직전에 다른 체크포인트로 추론 테스트를 했을 수 있으므로 완전히 새로 시작)
for _name in ["model", "base_model", "optimizer", "scheduler", "scaler"]:
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()

RESUME_FROM = "/content/qwen3_vl_8b_lora_epoch3"  # 가장 성능 좋았던 체크포인트
ADDITIONAL_EPOCHS = 2                              # epoch4, epoch5 추가 학습
RESUME_LR = 5e-5                                   # 이미 수렴된 지점에서 이어가는 것이라 처음(1e-4)보다 낮게
START_EPOCH = 3                                    # 체크포인트 번호를 이어가기 위한 기준(마지막으로 완료된 epoch 수)

# base_model 새로 로드 (오염 없는 깨끗한 상태)
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
base_model.enable_input_require_grads()

# 체크포인트의 LoRA 어댑터를 "학습 가능한" 상태로 로드 (추론용 로드와 달리 is_trainable=True 필수)
model = PeftModel.from_pretrained(base_model, RESUME_FROM, is_trainable=True)
model = model.to(device)
model.train()
model.print_trainable_parameters()

# 새 optimizer + 이번 추가 학습 분량(ADDITIONAL_EPOCHS)에 맞춘 새 스케줄러
# (기존 스케줄러는 3 epoch 동안 학습률이 0으로 수렴하도록 짜여 있어 재사용하면 사실상 학습이 안 됨)
optimizer = torch.optim.AdamW(model.parameters(), lr=RESUME_LR)
num_training_steps = ADDITIONAL_EPOCHS * math.ceil(len(train_loader)/GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(optimizer, int(num_training_steps*0.03), num_training_steps)
scaler = torch.amp.GradScaler('cuda', enabled=True)

print(f"이어서 학습 준비 완료: {RESUME_FROM} -> epoch{START_EPOCH+1}~{START_EPOCH+ADDITIONAL_EPOCHS} 학습 예정")

In [ ]:
# 추가 학습 루프 (기존 학습 루프와 동일한 구조, epoch 번호만 START_EPOCH부터 이어감)
global_step = 0
for extra_epoch in range(ADDITIONAL_EPOCHS):
    epoch = START_EPOCH + extra_epoch  # 실제 누적 epoch (0-indexed): 3, 4, ...
    running = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")
    for step, batch in enumerate(progress_bar, start=1):
        batch = {k:v.to(device) for k,v in batch.items()}
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()
        del batch, outputs, loss

        if step % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            avg_loss = running / GRAD_ACCUM
            progress_bar.set_postfix({"loss": f"{avg_loss:.3f}"})
            running = 0.0

            # 에폭 중간 진행상황 저장 (OOM/크래시로 통째로 날아가는 것 방지)
            if global_step % SAVE_EVERY_N_STEPS == 0:
                step_dir = f"{SAVE_DIR}_epoch{epoch+1}_step{global_step}"
                model.save_pretrained(step_dir)
                print("Saved mid-epoch checkpoint:", step_dir)
                torch.cuda.empty_cache()

    model.eval()
    val_loss = 0.0
    val_steps = 0
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid]", unit="batch"):
            vb = {k:v.to(device) for k,v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1
    print(f"[Epoch {epoch+1}] valid loss {val_loss/val_steps:.4f}")
    model.train()

    # epoch마다 체크포인트 저장
    epoch_dir = f"{SAVE_DIR}_epoch{epoch+1}"
    model.save_pretrained(epoch_dir)
    processor.save_pretrained(epoch_dir)
    print("Saved checkpoint:", epoch_dir)

print(f"추가 학습 완료: epoch{START_EPOCH+1}~epoch{START_EPOCH+ADDITIONAL_EPOCHS} 체크포인트 저장됨")
print("이 체크포인트들로도 추론 셀의 CHECKPOINT_DIR/RESUME_FROM 경로를 바꿔서 리더보드 점수 비교해보세요.")

# 오답 분석 (valid_subset 기준)

valid_subset은 gold answer가 있으므로, 실제 정답률과 오답 패턴을 직접 확인할 수 있습니다.

- 단순히 맞았다/틀렸다가 아니라, 모델이 **확신을 갖고 틀렸는지(진짜 오답 가능성)** vs **애매하게 갈렸는지(라벨 자체가 애매할 가능성)** 를 확률로 구분합니다
- 질문 유형별(개수/재질/색상/기타) 정답률도 함께 확인합니다
- 확신을 갖고 틀린 케이스는 이미지를 직접 띄워서 눈으로 검증합니다

현재 메모리에 있는 `model`을 그대로 사용하므로, 원하는 체크포인트로 먼저 로드해둔 상태에서 이 셀을 실행하세요.

In [ ]:
import gc
del model, base_model
gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)

from peft import PeftModel
CHECKPOINT_DIR = "/content/qwen3_vl_8b_lora_epoch3"
model = PeftModel.from_pretrained(base_model, CHECKPOINT_DIR)
model = model.to(device)
model.eval()

# 추론에 필요한 letter_ids/letter_id_tensor가 없으면(=추론 셀을 아직 안 돌렸으면) 여기서 생성
if "letter_id_tensor" not in globals():
    def build_letter_token_ids(processor):
        dummy_messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[{"type":"text","text":"dummy"}]},
        ]
        prefix_text = processor.apply_chat_template(dummy_messages, tokenize=False, add_generation_prompt=True)
        prefix_ids = processor.tokenizer(prefix_text, add_special_tokens=False)["input_ids"]
        letter_ids = {}
        for letter in ["a","b","c","d"]:
            full_ids = processor.tokenizer(prefix_text + letter, add_special_tokens=False)["input_ids"]
            assert full_ids[:len(prefix_ids)] == prefix_ids
            letter_ids[letter] = full_ids[len(prefix_ids):][0]
        return letter_ids
    letter_ids = build_letter_token_ids(processor)
    letters_order = ["a","b","c","d"]
    letter_id_tensor = torch.tensor([letter_ids[l] for l in letters_order], device=device)

# valid_subset에 대해 확률분포까지 기록하며 추론
VALID_INFER_BATCH_SIZE = 8
processor.tokenizer.padding_side = "left"

valid_eval = valid_subset.reset_index(drop=True).copy()
all_probs = []

with torch.no_grad():
    for start in tqdm(range(0, len(valid_eval), VALID_INFER_BATCH_SIZE), desc="Valid Analysis", unit="batch"):
        chunk = valid_eval.iloc[start:start+VALID_INFER_BATCH_SIZE]
        texts, images = [], []
        for _, row in chunk.iterrows():
            img = Image.open(row["path"]).convert("RGB")
            user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])
            messages = [
                {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
                {"role":"user","content":[
                    {"type":"image","image":img},
                    {"type":"text","text":user_text}
                ]}
            ]
            texts.append(processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
            images.append(img)

        inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(device)
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            out = model(**inputs)
        last_logits = out.logits[:, -1, :]
        probs = torch.softmax(last_logits[:, letter_id_tensor].float(), dim=-1)
        all_probs.extend(probs.cpu().tolist())
        del inputs, out, last_logits, probs

processor.tokenizer.padding_side = "right"

letters_order = ["a","b","c","d"]
valid_eval["prob_a"] = [p[0] for p in all_probs]
valid_eval["prob_b"] = [p[1] for p in all_probs]
valid_eval["prob_c"] = [p[2] for p in all_probs]
valid_eval["prob_d"] = [p[3] for p in all_probs]
valid_eval["pred"] = [letters_order[max(range(4), key=lambda i: p[i])] for p in all_probs]
valid_eval["pred_conf"] = [max(p) for p in all_probs]
valid_eval["gold_conf"] = [p[letters_order.index(g)] for p, g in zip(all_probs, valid_eval["answer"])]
valid_eval["correct"] = (valid_eval["pred"] == valid_eval["answer"])

acc = valid_eval["correct"].mean()
print(f"\n=== 전체 정확도: {acc:.4f} ({valid_eval['correct'].sum()}/{len(valid_eval)}) ===\n")

# 질문 유형별 정답률
def categorize(q):
    if "몇 개" in q or "개수" in q:
        return "개수(counting)"
    if "재질" in q or "소재" in q:
        return "재질(material)"
    if "색" in q:
        return "색상(color)"
    if "종류" in q:
        return "종류(type)"
    return "기타(other)"

valid_eval["category"] = valid_eval["question"].apply(categorize)
cat_summary = valid_eval.groupby("category")["correct"].agg(["mean","count"]).sort_values("mean")
print("=== 질문 유형별 정답률 ===")
print(cat_summary)
print()

# 오답을 "확신을 갖고 틀림" vs "애매하게 갈림"으로 정렬
wrong = valid_eval[~valid_eval["correct"]].copy()
wrong["conf_gap"] = wrong["pred_conf"] - wrong["gold_conf"]
print(f"틀린 문제 수: {len(wrong)} / {len(valid_eval)}")


In [ ]:
# "확신을 갖고 틀림" (진짜 오답일 가능성 높음) - 이미지까지 직접 눈으로 확인
from IPython.display import display

N_SHOW = 6
confident_wrong = wrong.sort_values("conf_gap", ascending=False).head(N_SHOW)

print(f"=== 확신을 갖고 틀린 케이스 상위 {len(confident_wrong)}개 (모델이 틀렸다고 강하게 의심되는 것들) ===\n")
for _, row in confident_wrong.iterrows():
    print(f"[{row['id']}] {row['question']}")
    print(f"  (a){row['a']} (b){row['b']} (c){row['c']} (d){row['d']}")
    print(f"  정답={row['answer']}(확신 {row['gold_conf']:.2f})  예측={row['pred']}(확신 {row['pred_conf']:.2f})")
    display(Image.open(row["path"]).convert("RGB").resize((300, 300)))
    print("-" * 60)

# "애매하게 갈림" (모델도 헷갈림 - 라벨 자체가 애매하거나 진짜 어려운 문제일 가능성)
ambiguous = wrong.sort_values("conf_gap", ascending=True).head(N_SHOW)
print(f"\n=== 애매하게 갈린 케이스 상위 {len(ambiguous)}개 (모델 확신도 정답/오답이 비슷 - 라벨 자체 애매성 의심) ===\n")
for _, row in ambiguous.iterrows():
    print(f"[{row['id']}] {row['question']}")
    print(f"  (a){row['a']} (b){row['b']} (c){row['c']} (d){row['d']}")
    print(f"  정답={row['answer']}(확신 {row['gold_conf']:.2f})  예측={row['pred']}(확신 {row['pred_conf']:.2f})")
    display(Image.open(row["path"]).convert("RGB").resize((300, 300)))
    print("-" * 60)


# 고해상도 재학습 (IMAGE_SIZE 384 → 448)

오답 분석 결과, **개수 세기(counting) 문제**가 정답률 82.2%로 가장 낮고, 전체 문제의 35%를 차지할 만큼 비중도 커서 가장 큰 개선 여지가 있는 부분이었습니다. 애매하게 틀린 케이스들도 대부분 "1개 차이"로 헷갈리는 패턴이라, 해상도 부족으로 인한 시각적 디테일 손실이 원인으로 의심됩니다.

- 기존 epoch1~5 체크포인트(`_epoch1`~`_epoch5`)는 건드리지 않도록 **별도 경로(`NEW_SAVE_DIR`)에 저장**합니다
- `NEW_DEBUG_SUBSET=True`로 먼저 소규모(300개)로 돌려서 메모리(OOM 여부)와 속도를 확인한 뒤, `False`로 바꿔 전체 데이터로 본 학습을 진행하세요
- 해상도가 올라가면 이미지 토큰 수가 늘어나 메모리 사용량이 커지므로 `NEW_BATCH_SIZE=1`로 보수적으로 시작합니다 (드라이런에서 여유가 확인되면 상향 가능)

In [ ]:
# 고해상도 실험 설정
NEW_IMAGE_SIZE = 512
NEW_BATCH_SIZE = 3          # 해상도 상승으로 이미지 토큰 수 증가 -> 안전하게 시작 (여유되면 2로 상향)
NEW_GRAD_ACCUM = 4          # 실배치 1이라 grad accumulation으로 효과적 배치 확보
NEW_EPOCHS = 3              # 기존과 동일하게
NEW_SAVE_DIR = "/content/qwen3_vl_8b_lora_res448"  # 기존 epoch1~5 체크포인트를 덮어쓰지 않도록 별도 경로
NEW_DEBUG_SUBSET = False     # 먼저 소규모로 메모리/속도 확인 후 False로 바꿔서 본 학습
NEW_DEBUG_N = 300

print(f"고해상도 실험 설정: IMAGE_SIZE={NEW_IMAGE_SIZE}, BATCH_SIZE={NEW_BATCH_SIZE}, SAVE_DIR={NEW_SAVE_DIR}, DEBUG_SUBSET={NEW_DEBUG_SUBSET}")

In [ ]:
import gc

# 기존 모델/프로세서 정리 (해상도가 다른 processor로 완전히 새로 만들기 위해)
for _name in ["model", "base_model", "optimizer", "scheduler", "scaler", "processor"]:
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()

# 새 해상도로 프로세서 재생성
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=NEW_IMAGE_SIZE*NEW_IMAGE_SIZE,
    max_pixels=NEW_IMAGE_SIZE*NEW_IMAGE_SIZE,
    trust_remote_code=True,
)

# 모델도 base부터 완전히 새로 로드 (기존 LoRA 가중치와 섞이지 않도록)
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
base_model.enable_input_require_grads()

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_R*2,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(base_model, lora_config)
model = model.to(device)
model.train()
model.print_trainable_parameters()

In [ ]:
# 기존 train_subset(90% + dev 증강 포함)/valid_subset(10%)는 그대로 재사용, 새 processor로만 다시 인코딩
res_train_subset = train_subset.copy()
if NEW_DEBUG_SUBSET:
    res_train_subset = res_train_subset.sample(n=min(NEW_DEBUG_N, len(res_train_subset)), random_state=SEED).reset_index(drop=True)

res_train_ds = VQAMCDataset(res_train_subset, processor, train=True, shuffle_choices=True, seed=SEED)
res_valid_ds = VQAMCDataset(valid_subset, processor, train=True, shuffle_choices=False, seed=SEED)

res_train_loader = DataLoader(res_train_ds, batch_size=NEW_BATCH_SIZE, shuffle=True, collate_fn=DataCollator(processor, True), num_workers=0)
res_valid_loader = DataLoader(res_valid_ds, batch_size=NEW_BATCH_SIZE, shuffle=False, collate_fn=DataCollator(processor, True), num_workers=0)

print(f"res_train_subset: {len(res_train_subset)} (DEBUG_SUBSET={NEW_DEBUG_SUBSET}), valid_subset: {len(valid_subset)}")

In [ ]:
# 고해상도 학습 루프 (기존 학습 루프와 동일한 구조)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
num_training_steps = NEW_EPOCHS * math.ceil(len(res_train_loader)/NEW_GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(optimizer, int(num_training_steps*0.03), num_training_steps)
scaler = torch.amp.GradScaler('cuda', enabled=True)

global_step = 0
for epoch in range(NEW_EPOCHS):
    running = 0.0
    progress_bar = tqdm(res_train_loader, desc=f"[res{NEW_IMAGE_SIZE}] Epoch {epoch+1} [train]", unit="batch")
    for step, batch in enumerate(progress_bar, start=1):
        batch = {k:v.to(device) for k,v in batch.items()}
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / NEW_GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()
        del batch, outputs, loss

        if step % NEW_GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            avg_loss = running / NEW_GRAD_ACCUM
            progress_bar.set_postfix({"loss": f"{avg_loss:.3f}"})
            running = 0.0

            if global_step % 500 == 0:
                step_dir = f"{NEW_SAVE_DIR}_epoch{epoch+1}_step{global_step}"
                model.save_pretrained(step_dir)
                print("Saved mid-epoch checkpoint:", step_dir)
                torch.cuda.empty_cache()

    model.eval()
    val_loss = 0.0
    val_steps = 0
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
        for vb in tqdm(res_valid_loader, desc=f"[res{NEW_IMAGE_SIZE}] Epoch {epoch+1} [valid]", unit="batch"):
            vb = {k:v.to(device) for k,v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1
    print(f"[res{NEW_IMAGE_SIZE} Epoch {epoch+1}] valid loss {val_loss/val_steps:.4f}")
    model.train()

    epoch_dir = f"{NEW_SAVE_DIR}_epoch{epoch+1}"
    model.save_pretrained(epoch_dir)
    processor.save_pretrained(epoch_dir)
    print("Saved checkpoint:", epoch_dir)

if NEW_DEBUG_SUBSET:
    print(f"\n드라이런 완료 (해상도={NEW_IMAGE_SIZE}, {NEW_DEBUG_N}개 샘플). OOM 없이 끝까지 돌았는지, 걸린 시간을 확인하고")
    print("NEW_DEBUG_SUBSET=False로 바꾼 뒤 위의 '기존 모델/프로세서 정리' 셀부터 다시 실행해서 본 학습을 진행하세요.")
else:
    print(f"\n고해상도({NEW_IMAGE_SIZE}) 본 학습 완료. {NEW_SAVE_DIR}_epoch1~{NEW_EPOCHS} 체크포인트가 저장되었습니다.")
    print("추론 셀(CHECKPOINT_DIR 부분)에서 이 경로로 바꿔 리더보드 점수를 비교해보세요.")

# 16. CoT(사고 과정) 추론 실험 (counting 정확도 개선 시도)

오답분석 결과, 개수 세기(counting) 문제가 전체 오답의 대부분(74%)을 차지하고 있고, 해상도를 384→512로 올려도 counting 정확도는 거의 안 움직였습니다(82.2%→83.9%). 즉 counting은 해상도 문제가 아니라, 모델이 바로 글자 하나만 찍는 방식(생각할 공간 없이 로그확률 비교만 하는 구조)이라 생기는 한계일 가능성이 높습니다.

이 섹션은 **재학습 없이** 기존 체크포인트(res512 epoch2, 폴더명은 res448이지만 실제로는 512 해상도로 학습된 체크포인트임, 섬션 15 참고)로, 추론 시점에 짧은 CoT(마야하지 않고 먼저 세어보게 하는 추론)를 거치게 해서 counting 정확도가 개선되는지 검증합니다.

- **사전 조건**: 셀 26~28(오답 분석)을 먼저 실행해서 `valid_eval`(category/pred/correct 컴럼 포함)이 메모리에 있어야 함. 이때 `CHECKPOINT_DIR`은 epoch2(실제 리더보드 최고점)로 맞춰서 돌린 결과여야 이후 비교가 공정함
- 이 섹션은 소규모 검증용이라, 효과가 확인되면 test_df 전체 추론 파이프라인에 적용하는 추가 셀을 따로 작성해야 함 (아직 미구현)


In [ ]:
import gc
from peft import PeftModel

for _name in ["model", "base_model"]:
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()

# 주의: 폴더명은 res448이지만 NEW_SAVE_DIR 버그로 실제 내용물은 512 해상도로 학습된 체크포인트 (리더보드 0.91486, 최고점)
COT_CHECKPOINT_DIR = "/content/qwen3_vl_8b_lora_res448_epoch2"
COT_IMAGE_SIZE = 512

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=COT_IMAGE_SIZE*COT_IMAGE_SIZE,
    max_pixels=COT_IMAGE_SIZE*COT_IMAGE_SIZE,
    trust_remote_code=True,
)
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)
model = PeftModel.from_pretrained(base_model, COT_CHECKPOINT_DIR)
model = model.to(device)
model.eval()

def build_letter_token_ids(processor):
    dummy_messages = [
        {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
        {"role":"user","content":[{"type":"text","text":"dummy"}]},
    ]
    prefix_text = processor.apply_chat_template(dummy_messages, tokenize=False, add_generation_prompt=True)
    prefix_ids = processor.tokenizer(prefix_text, add_special_tokens=False)["input_ids"]
    letter_ids = {}
    for letter in ["a","b","c","d"]:
        full_ids = processor.tokenizer(prefix_text + letter, add_special_tokens=False)["input_ids"]
        assert full_ids[:len(prefix_ids)] == prefix_ids
        letter_ids[letter] = full_ids[len(prefix_ids):][0]
    return letter_ids

letter_ids = build_letter_token_ids(processor)
letters_order = ["a","b","c","d"]
letter_id_tensor = torch.tensor([letter_ids[l] for l in letters_order], device=device)
print("checkpoint loaded:", COT_CHECKPOINT_DIR, "| letter ids:", letter_ids)


In [ ]:
COT_SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "먼저 사진 속에서 질문과 관련된 물체를 하나씩 짚어가며 한두 문장으로 간단히 확인한 뒤, "
    "마지막 줄에 반드시 '정답: '을 쓰고 이어서 a, b, c, d 중 하나의 소문자 한 글자만 적으세요."
)

def build_cot_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "먼저 물체를 하나씩 세어보며 간단히 설명하고, 마지막 줄에 '정답: <a/b/c/d>' 형식으로 답하세요."
    )

@torch.no_grad()
def infer_batch_cot(rows, max_new_tokens=120):
    texts, images = [], []
    for _, row in rows.iterrows():
        img = Image.open(row["path"]).convert("RGB")
        user_text = build_cot_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])
        messages = [
            {"role":"system","content":[{"type":"text","text":COT_SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}
            ]}
        ]
        texts.append(processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
        images.append(img)

    processor.tokenizer.padding_side = "left"
    inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(device)

    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        gen_out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=processor.tokenizer.pad_token_id,
        )
    input_len = inputs["input_ids"].shape[1]
    reasoning_texts = processor.tokenizer.batch_decode(gen_out[:, input_len:], skip_special_tokens=True)

    # 2단계: reasoning + "정답: " 강제 이어붙이고 마지막 위치 로짓 비교
    stage2_texts = []
    for (_, row), reasoning in zip(rows.iterrows(), reasoning_texts):
        user_text = build_cot_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])
        messages = [
            {"role":"system","content":[{"type":"text","text":COT_SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":Image.open(row["path"]).convert("RGB")},
                {"type":"text","text":user_text}
            ]},
        ]
        prefix_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        stage2_texts.append(prefix_text + reasoning.strip() + "\n정답: ")

    inputs2 = processor(text=stage2_texts, images=images, padding=True, return_tensors="pt").to(device)
    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        out2 = model(**inputs2)
    last_logits = out2.logits[:, -1, :]
    probs = torch.softmax(last_logits[:, letter_id_tensor].float(), dim=-1)
    preds = [letters_order[i] for i in probs.argmax(dim=-1).tolist()]

    processor.tokenizer.padding_side = "right"
    return preds, probs.cpu().tolist(), reasoning_texts


In [ ]:
# valid_eval은 셀 26~28(오답 분석, CHECKPOINT_DIR=epoch2)을 먼저 실행해야 생김. 없으면 여기서 멈춤.
assert "valid_eval" in globals(), "먼저 셀 26~28(오답 분석)을 CHECKPOINT_DIR=epoch2로 실행해서 valid_eval을 만들어주세요."

counting_wrong = valid_eval[(valid_eval["category"]=="개수(counting)") & (~valid_eval["correct"])].reset_index(drop=True)
print(f"기존 방식으로 틀린 counting 문제: {len(counting_wrong)}개")

COT_BATCH_SIZE = 4
fixed, still_wrong = 0, 0

for start in tqdm(range(0, len(counting_wrong), COT_BATCH_SIZE), desc="CoT retry"):
    chunk = counting_wrong.iloc[start:start+COT_BATCH_SIZE]
    preds, probs, reasonings = infer_batch_cot(chunk)
    for (_, row), pred, reasoning in zip(chunk.iterrows(), preds, reasonings):
        is_correct = (pred == row["answer"])
        fixed += is_correct
        still_wrong += (not is_correct)
        status = "✅고침" if is_correct else "❌여전히틀림"
        print(f"[{row['id']}] 정답={row['answer']} 기존예측={row['pred']} CoT예측={pred} {status}")
        print(f"  reasoning: {reasoning.strip()[:200]}")
        print("-"*60)

print(f"\nCoT로 새로 맞춘 것: {fixed} / {len(counting_wrong)}")


### 16-1. CoT 회귀(regression) 체크

앞서 8/29를 고친 건 "기존에 틀린 것만" 다시 풀어본 결과라, **기존에 맞았던 151개를 CoT로 다시 풀면 몇 개나 새로 틀리는지**는 아직 모릅니다. 이걸 확인해야 순이익을 판단할 수 있습니다. 전체 151개를 다 돌리면 시간이 오래 걸려서(배치당 ~10초), 우선 랜덤 샘플(기본 50개)로 대략적인 회귀율을 추정합니다.


In [ ]:
import random

correct_counting = valid_eval[(valid_eval["category"]=="개수(counting)") & (valid_eval["correct"])].reset_index(drop=True)
print(f"기존 방식으로 맞았던 counting 문제: {len(correct_counting)}개")

REGRESSION_SAMPLE_N = 50  # 전체(151개) 다 돌리고 싶으면 len(correct_counting)로 바꿀것
rng = random.Random(SEED)
sample_idx = rng.sample(range(len(correct_counting)), min(REGRESSION_SAMPLE_N, len(correct_counting)))
regression_sample = correct_counting.iloc[sample_idx].reset_index(drop=True)
print(f"회귀 체크용 샘플: {len(regression_sample)}개")

COT_BATCH_SIZE = 4
regressed, still_correct = 0, 0
regressed_rows = []

for start in tqdm(range(0, len(regression_sample), COT_BATCH_SIZE), desc="CoT regression check"):
    chunk = regression_sample.iloc[start:start+COT_BATCH_SIZE]
    preds, probs, reasonings = infer_batch_cot(chunk)
    for (_, row), pred, reasoning in zip(chunk.iterrows(), preds, reasonings):
        is_correct = (pred == row["answer"])
        still_correct += is_correct
        regressed += (not is_correct)
        if not is_correct:
            regressed_rows.append(row["id"])
            print(f"[{row['id']}] 정답={row['answer']} 기존예측={row['pred']}(맞음) CoT예측={pred} ⚠️회귀(새로틀림)")
            print(f"  reasoning: {reasoning.strip()[:200]}")
            print("-"*60)

regression_rate = regressed / len(regression_sample)
est_regressed_full = regression_rate * len(correct_counting)
net_estimate = len(correct_counting) - est_regressed_full + 8  # 8 = 앞서 CoT로 새로 맞춘 개수

print(f"\n샘플 {len(regression_sample)}개 중 CoT로 새로 틀린 것(회귀): {regressed}개 ({regression_rate*100:.1f}%)")
print(f"전체 {len(correct_counting)}개로 추정 시 예상 회귀: 약 {est_regressed_full:.1f}개")
print(f"순이익 추정: 기존 정답 {len(correct_counting)}개 - 추정회귀 {est_regressed_full:.1f}개 + CoT로 고친 8개 = 약 {net_estimate:.1f}개 (기존 151개 대비)")
print(f"추정 counting 정확도: {net_estimate:.1f}/180 = {net_estimate/180*100:.1f}% (기존 83.9% 대비)")


### 16-2. 신뢰도 기반 CoT 선택 적용

CoT를 counting 전체에 그대로 적용하면 순이익이 거의 0(오히려 약간 마이너스)이었습니다. 원인은 CoT가 **원래 맞던 답을 스스로 뒤집는 회귀** 사례들이었습니다. 이 섹션은 **CoT의 최종 확신도(prob)가 높을 때만 기존 답을 덮어쓰고, 낮으면 기존 답을 유지**하는 전략으로, "자신이 없는 CoT가 원래 맞던 답을 망치는" 케이스를 거르려내는가 검증합니다.

- **사전 조건**: 셀 26~28(오답 분석, CHECKPOINT_DIR=epoch2)을 먼저 실행해서 `valid_eval`이 있어야 함. 셀 35~36(CoT 추론 함수 `infer_batch_cot`)도 정의된 상태여야 함.
- counting 카테고리 전체(180개)를 CoT로 다시 돌리므로 약 8~9분 정도 소요됩니다.


In [ ]:
counting_full = valid_eval[valid_eval["category"]=="개수(counting)"].reset_index(drop=True)
print(f"counting 전체: {len(counting_full)}개 (CoT로 전부 재추론)")

COT_BATCH_SIZE = 4
cot_preds, cot_probs_max = [], []

for start in tqdm(range(0, len(counting_full), COT_BATCH_SIZE), desc="CoT full run (counting)"):
    chunk = counting_full.iloc[start:start+COT_BATCH_SIZE]
    preds, probs, reasonings = infer_batch_cot(chunk)
    cot_preds.extend(preds)
    cot_probs_max.extend([max(p) for p in probs])

counting_full["cot_pred"] = cot_preds
counting_full["cot_prob"] = cot_probs_max
counting_full["cot_correct"] = (counting_full["cot_pred"] == counting_full["answer"])

print(f"\nCoT 단독 정확도: {counting_full['cot_correct'].mean()*100:.1f}% ({counting_full['cot_correct'].sum()}/{len(counting_full)})")
print(f"기존(baseline) 정확도: {counting_full['correct'].mean()*100:.1f}% ({counting_full['correct'].sum()}/{len(counting_full)})")


In [ ]:
import numpy as np

print(f"{'threshold':>10} | {'final_acc':>10} | {'CoT 채택 수':>12}")
results = []
for th in np.arange(0.50, 1.00, 0.05):
    use_cot = counting_full["cot_prob"] >= th
    final_pred = np.where(use_cot, counting_full["cot_pred"], counting_full["pred"])
    final_correct = (final_pred == counting_full["answer"]).mean()
    results.append((th, final_correct, int(use_cot.sum())))
    print(f"{th:>10.2f} | {final_correct*100:>9.1f}% | {int(use_cot.sum()):>12d}")

best_th, best_acc, best_n = max(results, key=lambda r: r[1])
print(f"\n최적 threshold: {best_th:.2f} (counting 정확도 {best_acc*100:.1f}%, CoT 채택 {best_n}개/180)")
print(f"기존(baseline) 대비: {counting_full['correct'].mean()*100:.1f}% -> {best_acc*100:.1f}%")

# 최적 threshold를 전체 valid에 적용했을 때 전체 정확도 변화
overall_before = valid_eval["correct"].mean()
use_cot_best = counting_full["cot_prob"] >= best_th
updated_counting_pred = np.where(use_cot_best, counting_full["cot_pred"], counting_full["pred"])

updated = valid_eval.copy()
mask = (updated["category"]=="개수(counting)")
updated.loc[mask, "pred"] = updated_counting_pred
updated["correct"] = (updated["pred"] == updated["answer"])
overall_after = updated["correct"].mean()

print(f"\n전체 valid 정확도: {overall_before*100:.2f}% -> {overall_after*100:.2f}% (threshold={best_th:.2f} 적용 시)")


### 16-3. TTA (멀티 스케일 앙상블)

CoT는 별로였으니, 이번엔 "다른 해상도로 여러 번 다시 보고 확률을 평균내는" 방식으로 겹침/가림 문제를 완화하려는 시도입니다. 기존 CoT와 달리 생각(reasoning) 없이 기존 로그확률 비교 방식 그대로라 속도도 빠릅니다.

- 모델은 그대로(res512 epoch2 LoRA), **프로세서만 해상도별로 새로 만들어** 같은 이미지를 여러 픽셀 예산으로 인코딩함 (LoRA는 언어모델 쪽에만 있고 vision tower는 원래 dynamic resolution을 지원하므로 학습 때와 다른 해상도를 넣어도 동작함)
- 각 스케일의 softmax 확률을 평균내서 최종 예측
- **사전 조건**: 셀 35(체크포인트 로드 + letter_id_tensor)와 셀 26~28(오답 분석, `valid_eval`)이 먼저 실행된 상태여야 함


In [ ]:
@torch.no_grad()
def infer_batch_logprob(rows, tta_processor, batch_size=8):
    all_probs = []
    tta_processor.tokenizer.padding_side = "left"
    for start in range(0, len(rows), batch_size):
        chunk = rows.iloc[start:start+batch_size]
        texts, images = [], []
        for _, row in chunk.iterrows():
            img = Image.open(row["path"]).convert("RGB")
            user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])
            messages = [
                {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
                {"role":"user","content":[
                    {"type":"image","image":img},
                    {"type":"text","text":user_text}
                ]}
            ]
            texts.append(tta_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
            images.append(img)

        inputs = tta_processor(text=texts, images=images, padding=True, return_tensors="pt").to(device)
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            out = model(**inputs)
        last_logits = out.logits[:, -1, :]
        probs = torch.softmax(last_logits[:, letter_id_tensor].float(), dim=-1)
        all_probs.extend(probs.cpu().tolist())
        del inputs, out, last_logits, probs

    tta_processor.tokenizer.padding_side = "right"
    return all_probs


In [ ]:
import numpy as np

TTA_SCALES = [384, 448, 512, 576]  # 학습 해상도(512)를 중심으로 위아래 폭널링

scale_probs = {}
for size in TTA_SCALES:
    print(f"--- scale {size} 시작 ---")
    tta_processor = AutoProcessor.from_pretrained(
        MODEL_ID, min_pixels=size*size, max_pixels=size*size, trust_remote_code=True
    )
    probs = infer_batch_logprob(counting_full, tta_processor, batch_size=8)
    scale_probs[size] = np.array(probs)
    print(f"--- scale {size} 완료 ---")

avg_probs = np.mean([scale_probs[s] for s in TTA_SCALES], axis=0)
tta_pred_idx = avg_probs.argmax(axis=1)
tta_pred = [letters_order[i] for i in tta_pred_idx]

counting_full["tta_pred"] = tta_pred
counting_full["tta_correct"] = (counting_full["tta_pred"] == counting_full["answer"])

print(f"\nTTA(멀티스케일 평균, {TTA_SCALES}) counting 정확도: {counting_full['tta_correct'].mean()*100:.1f}% ({counting_full['tta_correct'].sum()}/{len(counting_full)})")
print(f"기존(512 단일) counting 정확도: {counting_full['correct'].mean()*100:.1f}% ({counting_full['correct'].sum()}/{len(counting_full)})")

# 전체 valid에 TTA 결과를 반영했을 때 전체 정확도
overall_before = valid_eval["correct"].mean()
updated_tta = valid_eval.copy()
mask = (updated_tta["category"]=="개수(counting)")
updated_tta.loc[mask, "pred"] = counting_full["tta_pred"].values
updated_tta["correct"] = (updated_tta["pred"] == updated_tta["answer"])
overall_after = updated_tta["correct"].mean()
print(f"\n전체 valid 정확도: {overall_before*100:.2f}% -> {overall_after*100:.2f}% (TTA 적용 시)")


### 16-3-1. TTA 결과가 모든 스케일에서 동일하게 나온 이유 진단

4개 스케일 평균이 baseline(512 단일)과 정확히 같게 151/180으로 나왔습니다. 이게 "해상도가 진짜 안 먹히는 문제"인지, 아니면 `min_pixels`/`max_pixels`가 실제로 적용이 안 되고 있는 버그인지 확인합니다.


In [ ]:
import numpy as np

print("=== 스칼별 단독 정확도 (평균내기 전) ===")
for size in TTA_SCALES:
    preds_at_scale = np.array([letters_order[i] for i in scale_probs[size].argmax(axis=1)])
    acc_at_scale = (preds_at_scale == counting_full["answer"].values).mean()
    print(f"scale {size}: {acc_at_scale*100:.1f}%")

print("\n=== 스케일 간 확률값 자체가 다른지 확인 ===")
diff_384_576 = np.abs(scale_probs[384] - scale_probs[576]).mean()
identical_384_576 = np.array_equal(scale_probs[384], scale_probs[576])
print(f"scale 384 vs 576 확률 평균 절대차: {diff_384_576:.6f}")
print(f"384와 576의 확률이 완전히 동일한가?: {identical_384_576}")

print("\n=== 실제 이미지 토큰 수(해상도 반영 여부) 확인 - 샘플 1장 ===")
sample_row = counting_full.iloc[0]
sample_img = Image.open(sample_row["path"]).convert("RGB")
print(f"원본 이미지 크기: {sample_img.size}")
for size in TTA_SCALES:
    p = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=size*size, max_pixels=size*size, trust_remote_code=True)
    out = p(text=["dummy"], images=[sample_img], return_tensors="pt")
    grid = out.get("image_grid_thw")
    print(f"scale {size}: input_ids 길이={out['input_ids'].shape[1]}, image_grid_thw={grid}")


# 17. 최종 test 추론 파이프라인 (baseline + counting 신뢰도 게이팅 CoT)

지금까지의 실험 결과를 종합해서 구성한 최종 추론 파이프라인입니다:

- **전체 문항**: 기존 로그확률 비교 방식(baseline) 그대로 사용 (이미 96%+ 수준이라 CoT/TTA 둘 다 순이익 없음을 확인함)
- **개수(counting) 문항만**: CoT로 재추론 후, CoT 확신도가 `FINAL_COT_THRESHOLD` 이상일 때만 baseline 답을 CoT 답으로 덮어쓰기 (셀 42 threshold 스윗에서 안정적으로 85.0%를 짐은 0.60~0.75 구간의 중간값인 0.65 선택 — 0.90은 단일 스파이크로 의심되어 제외)
- TTA(멀티스케일)는 순이익 없음이 확인되어 최종 파이프라인에서 제외

- **사전 조건**: 셀 35(체크포인트 로드, `model`/`processor`/`letter_id_tensor`)와 셀 36(`infer_batch_cot`)이 먼저 실행된 상태여야 함


In [ ]:
FINAL_COT_THRESHOLD = 0.65  # 셀 42 threshold 스윗의 안정적인 구간(0.60~0.75) 중간값

def categorize(q):
    if "목 개" in q or "개수" in q:
        return "개수(counting)"
    if "재질" in q or "소재" in q:
        return "재질(material)"
    if "색" in q:
        return "색상(color)"
    if "종류" in q:
        return "종류(type)"
    return "기타(other)"

test_df["category"] = test_df["question"].apply(categorize)
print(test_df["category"].value_counts())

# 1단계: 전체 test_df에 baseline 로그확률 추론
INFER_BATCH_SIZE = 8
processor.tokenizer.padding_side = "left"

baseline_preds = []
with torch.no_grad():
    for start in tqdm(range(0, len(test_df), INFER_BATCH_SIZE), desc="Baseline inference (test)"):
        chunk = test_df.iloc[start:start+INFER_BATCH_SIZE]
        texts, images = [], []
        for _, row in chunk.iterrows():
            img = Image.open(row["path"]).convert("RGB")
            user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])
            messages = [
                {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
                {"role":"user","content":[
                    {"type":"image","image":img},
                    {"type":"text","text":user_text}
                ]}
            ]
            texts.append(processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
            images.append(img)

        inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(device)
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            out = model(**inputs)
        last_logits = out.logits[:, -1, :]
        probs = torch.softmax(last_logits[:, letter_id_tensor].float(), dim=-1)
        best_idx = probs.argmax(dim=-1).tolist()
        baseline_preds.extend(letters_order[i] for i in best_idx)
        del inputs, out, last_logits, probs

processor.tokenizer.padding_side = "right"
test_df["baseline_pred"] = baseline_preds
print(f"\nbaseline 추론 완료. 예측 분포: {test_df['baseline_pred'].value_counts().to_dict()}")


In [ ]:
# 2단계: counting 문항만 CoT로 재추론 후 신뢰도 게이팅으로 덮어쓰기
counting_test = test_df[test_df["category"]=="개수(counting)"].copy()  # 원래 index 유지
print(f"test 중 counting 문항: {len(counting_test)}개")

COT_BATCH_SIZE = 4
cot_preds_test, cot_probs_test = [], []

for start in tqdm(range(0, len(counting_test), COT_BATCH_SIZE), desc="CoT inference (test counting)"):
    chunk = counting_test.iloc[start:start+COT_BATCH_SIZE]
    preds, probs, reasonings = infer_batch_cot(chunk)
    cot_preds_test.extend(preds)
    cot_probs_test.extend([max(p) for p in probs])

counting_test["cot_pred"] = cot_preds_test
counting_test["cot_prob"] = cot_probs_test

final_preds = test_df["baseline_pred"].copy()
use_cot = counting_test["cot_prob"] >= FINAL_COT_THRESHOLD
final_preds.loc[counting_test.index[use_cot]] = counting_test.loc[use_cot, "cot_pred"]

n_changed = int((final_preds != test_df["baseline_pred"]).sum())
print(f"\nCoT로 덮어쓴 문항 수: {use_cot.sum()} / {len(counting_test)} (threshold={FINAL_COT_THRESHOLD})")
print(f"baseline 대비 실제로 답이 바뀌 문항 수: {n_changed}")


In [ ]:
submission = pd.DataFrame({"id": test_df["id"], "answer": final_preds})
submission.to_csv("/content/submission.csv", index=False)
print("Saved /content/submission.csv")

print("\n=== 최종 예측 분포 (특정 글자 쓰임 없는지 sanity check) ===")
print(submission["answer"].value_counts())

print("\n=== baseline만 쓨을 때와의 분포 비교 ===")
print(test_df["baseline_pred"].value_counts())
